# Khanh 27/09: three baselines on Qwen2-VL-7B
Select an **A100 GPU** runtime, then run every cell. The output directory lives on Drive so the run can resume after a disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi


In [ ]:
!git clone --branch results/qwen2vl-jina4-four-datasets-2026-09-14 --single-branch https://github.com/NguyenKhanh2603/Uncertainty-Aware-Iterative-RAG.git /content/Uncertainty-Aware-Iterative-RAG
%cd /content/Uncertainty-Aware-Iterative-RAG/research/internal_state_rag/protocols/khanh_27_09_colab_three_baselines
!pip install -q -r requirements.txt


In [ ]:
from huggingface_hub import hf_hub_download
import hashlib, json, pathlib, tarfile
meta = json.loads(pathlib.Path('DATA_BUNDLE.json').read_text())
archive = hf_hub_download(repo_id=meta['repo_id'], repo_type='dataset', filename=meta['filename'], local_dir='.')
digest = hashlib.sha256(pathlib.Path(archive).read_bytes()).hexdigest()
assert digest == meta['sha256'], (digest, meta['sha256'])
with tarfile.open(archive, 'r:gz') as handle:
    handle.extractall('.')
print(meta)


In [ ]:
OUTPUT_DIR = '/content/drive/MyDrive/khanh_27_09_three_baselines_colab'
!python run_colab.py --data-root data_bundle --output-dir "{OUTPUT_DIR}"


In [ ]:
import json, pathlib, shutil, subprocess
summary = json.loads((pathlib.Path(OUTPUT_DIR) / 'summary.json').read_text())
assert summary['status'] == 'complete'
assert sum(1 for _ in open(pathlib.Path(OUTPUT_DIR) / 'progress_log.jsonl')) == 1200
subprocess.run(['python', 'merge_resource_columns.py', '--csv', f'{OUTPUT_DIR}/final_downstream_results.csv', '--output', f'{OUTPUT_DIR}/REPORT_WITH_RESOURCE_METRICS.md'], check=True)
archive = shutil.make_archive('/content/khanh_27_09_colab_results', 'zip', OUTPUT_DIR)
from google.colab import files
files.download(archive)
